# TODO: 
Format Everything (mostly according to the project outline)

## Libraries & Packages

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

## Data Loading

In [2]:
# Yearly appropriations for NASA compared to all other organizations
fed_budgets_raw = pd.read_csv("data/federal_budgets.csv")

# Yearly allocations to the constituent divisions of NASA's Science Directorate
division_budgets_raw = pd.read_csv("data/nasa_division_budgets.csv")

# Mean cont of sunspots per month
sunspots_raw = pd.read_json("data/sunspot_records.json")

# Date limits of each solar cycle
solar_cycle_raw = pd.DataFrame({
    'cycle_num' : [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25],
    'start_date' : 	["1755-02", "1766-06", "1775-06", "1784-09", "1798-04", "1810-07",
                     "1823-05", "1833-11", "1843-07", "1855-12", "1867-03", "1878-12",
                     "1890-03", "1902-01", "1913-07", "1923-08", "1933-09", "1944-02",
                     "1954-04", "1964-10", "1976-03", "1986-09", "1996-08", "2008-12", "2019-12"],
    'end_date' : ["1766-06", "1775-06", "1784-09", "1798-04", "1810-07",
                  "1823-05", "1833-11", "1843-07", "1855-12", "1867-03", "1878-12",
                  "1890-03", "1902-01", "1913-07", "1923-08", "1933-09", "1944-02",
                  "1954-04", "1964-10", "1976-03", "1986-09", "1996-08", "2008-12", "2019-12", "2030-06"],
    'expected_start' : ["1755-02", "1766-02", "1777-02", "1788-02", "1799-02", "1810-02", "1821-02", "1832-02",
                        "1843-02", "1854-02", "1865-02", "1876-02", "1887-02", "1898-02", "1909-02", "1920-02",
                        "1931-02", "1942-02", "1953-02", "1964-02", "1975-02", "1986-02", "1997-02", "2008-02","2019-02"],
    'expected_end' : ["1766-02", "1777-02", "1788-02", "1799-02", "1810-02", "1821-02", "1832-02",
                      "1843-02", "1854-02", "1865-02", "1876-02", "1887-02", "1898-02", "1909-02", "1920-02",
                      "1931-02", "1942-02", "1953-02", "1964-02", "1975-02", "1986-02", "1997-02", "2008-02","2019-02", "2030-02"]
})



## Cleaning & Transformations

### Transformation Helper Functions

In [3]:
# Given a date, returns which observed solar cycle that date is in
def getObservedCycle(date, df) :
    
    # Check for which solar cycle a date falls in
    mask = (df['start_date'] <= date) & (date < df['end_date'])
    match = df[mask]
    
    if not match.empty:
        return match.iloc[0]['cycle_num'] 
    
    # Either the date is in the future, or it's before the first official solar cycle in 1755
    return np.nan


# Given a date, returns the expected solar cycle assuming a consistent 11-year cycle
# (Maybe redundant to have a slightly tweaked copy of getObservedCycle, but I think having separate funcs improves legibility)
def getExpectedCycle(date, df) :

    mask = (df['expected_start'] <= date) & (date < df['expected_end'])
    match = df[mask]
    
    if not match.empty:
        return match.iloc[0]['cycle_num'] 
    
    # Either the date is in the future, or it's before the first official solar cycle in 1755
    return np.nan


### Solar Cycle Dataset

In [4]:
solar_cycle = solar_cycle_raw.copy()

# Cast as datetime
solar_cycle['start_date'] = pd.to_datetime(solar_cycle['start_date'])
solar_cycle['end_date'] = pd.to_datetime(solar_cycle['end_date'])
solar_cycle['expected_start'] = pd.to_datetime(solar_cycle['expected_start'])
solar_cycle['expected_end'] = pd.to_datetime(solar_cycle['expected_end'])


### Sunspots Dataset

In [5]:
sunspots = sunspots_raw.copy()

# Reformat columns
sunspots = sunspots.drop(labels=['smoothed_ssn', 'observed_swpc_ssn', 'smoothed_swpc_ssn', 'f10.7', 'smoothed_f10.7'], axis=1)
sunspots = sunspots.rename(columns={'time-tag' : 'date'})

# Cast as datetime
sunspots['date'] = pd.to_datetime(sunspots['date'])

# Set solar Observed and Expected cycle numbers
sunspots['cycle_obsv'] = sunspots['date'].apply(lambda x: getObservedCycle(x, solar_cycle)).astype('Int64')
sunspots['cycle_expc'] = sunspots['date'].apply(lambda x: getExpectedCycle(x, solar_cycle)).astype('Int64')


# Drop rows from before the first recorded solar cycle
sunspots = sunspots.dropna().reset_index(drop=True)

# Assign rows a government fiscal year 
# (runs from October of the preceding calendar year through September of the current calendar year)
# Example: December 2025 is in US Fiscal Year 2026
sunspots['gov_fiscal_year'] = sunspots['date'].dt.to_period('Y-SEP')
sunspots['gov_fiscal_year'] = sunspots['gov_fiscal_year'].apply(lambda x : x.year).astype('int64')

# An 11-year solar cycle is supposed to start and end in periods of low solar activity
# Assign 'high' and 'low' flags accordingly
sunspots['activity_expc'] = sunspots.groupby('cycle_expc')['date'].transform(
    lambda x: pd.cut(x, bins=4, ordered=False, labels=['low', 'high', 'high', 'low']))

# Save a copy o
#sunspots_nasa = sunspots.loc[sunspots['date'].dt.year >= 1960].copy().reset_index(drop=True)


### Federal Budgets Dataset

In [7]:

fed_budgets = fed_budgets_raw.copy()

# Cast budget data as numeric
fed_budgets['budget_nominal'] = fed_budgets['budget_nominal'].str.replace(',', '')
fed_budgets['budget_nominal'] = pd.to_numeric(fed_budgets['budget_nominal']).astype('float64')
fed_budgets['budget_real'] = fed_budgets['budget_real'].str.replace(',', '')
fed_budgets['budget_real'] = pd.to_numeric(fed_budgets['budget_real']).astype('float64')

# Create datetime attributes
fed_budgets['gfy_start'] = pd.to_datetime(fed_budgets['gov_fiscal_year'], format='%Y') - pd.DateOffset(months=3)
fed_budgets['gfy_end'] = fed_budgets['gfy_start'] + pd.DateOffset(months=12)

# Merge sunspot data onto fed_budgets
sunspots_fed = sunspots.copy()
sunspots_fed = sunspots_fed.groupby('gov_fiscal_year')['ssn'].mean() # Mean sunspots in fiscal year
fed_budgets = pd.merge(fed_budgets, sunspots_fed, on='gov_fiscal_year')



### Science Divisions Budgets Dataset

In [8]:
division_budgets = division_budgets_raw.copy()

# Cast budget data as numeric
division_budgets['budget_nominal'] = division_budgets['budget_nominal'].str.replace(',', '')
division_budgets['budget_nominal'] = pd.to_numeric(division_budgets['budget_nominal']).astype('float64')
division_budgets['budget_real'] = division_budgets['budget_real'].str.replace(',', '')
division_budgets['budget_real'] = pd.to_numeric(division_budgets['budget_real']).astype('float64')

# Create datetime attributes
division_budgets['gfy_start'] = pd.to_datetime(division_budgets['gov_fiscal_year'], format='%Y') - pd.DateOffset(months=3)
division_budgets['gfy_end'] = division_budgets['gfy_start'] + pd.DateOffset(months=12)

# Merge sunspot data onto division_budgets
sunspots_div = sunspots.copy()
sunspots_div = sunspots_div.groupby('gov_fiscal_year')['ssn'].mean() # Mean sunspots in fiscal year
division_budgets = pd.merge(division_budgets, sunspots_fed, on='gov_fiscal_year')